In [ ]:
import pandas as pd
import numpy as np
import requests
import zipfile
import os
import re
import sys
import logging
import warnings
import contextlib
import yfinance as yf
from datetime import datetime, timedelta
from tqdm import tqdm
from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, Alignment, PatternFill

# ==============================================================================
# ⚙️ 1. CONFIGURAÇÕES
# ==============================================================================

ANO_INICIO = 2015
ANO_FIM = 2025

BASE_DIR = os.path.join(os.getcwd(), "pipeline_cvm_final")
RAW_DIR = os.path.join(BASE_DIR, "temp_cache") 
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs")

CAMINHO_TICKERS = r"BASE_EMPRESAS_TICKERS.csv" 
CVM_BASE_URL = "https://dados.cvm.gov.br/dados/CIA_ABERTA/DOC/"

# ==============================================================================
# 🛠️ 2. FUNÇÕES UTILITÁRIAS
# ==============================================================================

def setup_directories():
    for path in [RAW_DIR, OUTPUT_DIR]:
        os.makedirs(path, exist_ok=True)
    print(f"📂 Diretórios configurados em: {BASE_DIR}")

def limpar_cnpj(cnpj):
    return re.sub(r'\D', '', str(cnpj))

def converter_escala(valor, escala):
    if pd.isna(valor): return 0.0
    if escala == 'MIL': return valor * 1000.0
    return valor

@contextlib.contextmanager
def suppress_output():
    with open(os.devnull, 'w') as devnull:
        old_stdout = sys.stdout
        old_stderr = sys.stderr
        try:
            sys.stdout = devnull
            sys.stderr = devnull
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr

def download_zip_temporario(tipo, ano):
    filename = f"{tipo.lower()}_cia_aberta_{ano}.zip"
    url = f"{CVM_BASE_URL}{tipo}/DADOS/{filename}"
    file_path = os.path.join(RAW_DIR, filename)
    
    if os.path.exists(file_path):
        return file_path
        
    try:
        r = requests.get(url, stream=True)
        if r.status_code == 200:
            total_size = int(r.headers.get('content-length', 0))
            with open(file_path, 'wb') as f, tqdm(desc=filename, total=total_size, unit='iB', unit_scale=True) as bar:
                for chunk in r.iter_content(chunk_size=1024):
                    size = f.write(chunk)
                    bar.update(size)
            return file_path
        else:
            return None
    except Exception as e:
        print(f"   ❌ Erro download {filename}: {e}")
        return None

def limpar_cache_ano(ano):
    for f in os.listdir(RAW_DIR):
        if str(ano) in f and f.endswith('.zip'):
            try:
                os.remove(os.path.join(RAW_DIR, f))
            except:
                pass

def carregar_tickers_e_tipos(caminho):
    print(f"\n📖 Carregando Mapeamento de Tickers e Tipos...")
    if not os.path.exists(caminho):
        print("⚠️ Arquivo de tickers não encontrado.")
        return {}, {}
        
    encodings = ['utf-8-sig', 'latin1', 'cp1252']
    for enc in encodings:
        try:
            df = pd.read_csv(caminho, sep=';', dtype=str, encoding=enc)
            df.columns = df.columns.str.upper().str.strip()
            
            if 'CNPJ' in df.columns and 'TICKER' in df.columns:
                df['CNPJ_CLEAN'] = df['CNPJ'].apply(limpar_cnpj)
                df['TICKER'] = df['TICKER'].str.strip().str.upper()
                
                # Tratar coluna Tipo (se existir)
                if 'TIPO' not in df.columns:
                    df['TIPO'] = 'N/A'
                else:
                    df['TIPO'] = df['TIPO'].str.strip()
                
                map_tickers = dict(zip(df['CNPJ_CLEAN'], df['TICKER']))
                map_tipos = dict(zip(df['CNPJ_CLEAN'], df['TIPO']))
                
                return map_tickers, map_tipos
        except:
            continue
    return {}, {}

# ==============================================================================
# 🧮 3. LEITURA DE ARQUIVOS (DIRETO DO ZIP)
# ==============================================================================

def ler_csv_do_zip(path_zip, nome_csv_parcial, colunas):
    if not path_zip or not os.path.exists(path_zip):
        return pd.DataFrame()
    
    try:
        with zipfile.ZipFile(path_zip, 'r') as z:
            arquivos_no_zip = z.namelist()
            arquivo_alvo = next((f for f in arquivos_no_zip if nome_csv_parcial.lower() in f.lower() and f.endswith('.csv')), None)
            
            if not arquivo_alvo:
                return pd.DataFrame()

            with z.open(arquivo_alvo) as f:
                cabecalho = pd.read_csv(f, sep=';', encoding='latin1', nrows=0)
                cols_arquivo = cabecalho.columns.tolist()
                cols_uteis = [c for c in colunas if c in cols_arquivo]
            
            with z.open(arquivo_alvo) as f:
                df = pd.read_csv(f, sep=';', encoding='latin1', usecols=cols_uteis, low_memory=False)

        if 'ORDEM_EXERC' in df.columns:
            df = df[df['ORDEM_EXERC'] == 'ÚLTIMO']
            
        if 'DT_FIM_EXERC' in df.columns:
            df['DT_FIM_EXERC'] = pd.to_datetime(df['DT_FIM_EXERC'])
        if 'DT_INI_EXERC' in df.columns:
            df['DT_INI_EXERC'] = pd.to_datetime(df['DT_INI_EXERC'])
        
        if 'VL_CONTA' in df.columns and 'ESCALA_MOEDA' in df.columns:
            df['VL_REAL'] = df.apply(lambda x: converter_escala(x['VL_CONTA'], x['ESCALA_MOEDA']), axis=1)
        
        if 'CNPJ_CIA' in df.columns:
            df['CNPJ_CLEAN'] = df['CNPJ_CIA'].apply(limpar_cnpj)
        
        return df
    except Exception as e:
        return pd.DataFrame()

# ==============================================================================
# 🧠 4. ENGINE DE PROCESSAMENTO E CÁLCULO CONTÁBIL
# ==============================================================================

def extrair_patrimonio(df_bpa, df_bpp, data):
    bpa_d = df_bpa[df_bpa['DT_FIM_EXERC'] == data]
    bpp_d = df_bpp[df_bpp['DT_FIM_EXERC'] == data]
    
    if bpa_d.empty or bpp_d.empty: return pd.DataFrame()
    
    ativo = bpa_d[bpa_d['CD_CONTA'] == '1'].groupby('CNPJ_CLEAN')['VL_REAL'].sum()
    caixa = bpa_d[bpa_d['CD_CONTA'] == '1.01.01'].groupby('CNPJ_CLEAN')['VL_REAL'].sum()
    pl = bpp_d[bpp_d['CD_CONTA'] == '2.03'].groupby('CNPJ_CLEAN')['VL_REAL'].sum()
    divida = bpp_d[bpp_d['CD_CONTA'].isin(['2.01.04', '2.02.01'])].groupby('CNPJ_CLEAN')['VL_REAL'].sum()
    
    res = pd.DataFrame({
        'Ativo Total': ativo,
        'Caixa': caixa,
        'Patrimonio Liquido': pl,
        'Divida Bruta': divida
    }).reset_index()
    
    res['Divida Liquida'] = res['Divida Bruta'].fillna(0) - res['Caixa'].fillna(0)
    return res

def agrupar_contas_resultado(df_conta, col_valor):
    """
    Seleciona o Lucro Líquido com hierarquia para evitar duplicação.
    Ordem: 3.11 > 3.11.01 > 3.07 > 3.09
    """
    resultados = []
    
    for cnpj, grupo in df_conta.groupby('CNPJ_CLEAN'):
        # Lógica Hierárquica para Lucro Líquido
        ll = 0
        
        # 1. Tenta conta 3.11 (Geral)
        val_311 = grupo[grupo['CD_CONTA'] == '3.11'][col_valor].sum()
        if val_311 != 0:
            ll = val_311
        else:
            # 2. Tenta conta 3.11.01 (Atribuído a Sócios Controladores)
            val_31101 = grupo[grupo['CD_CONTA'] == '3.11.01'][col_valor].sum()
            if val_31101 != 0:
                ll = val_31101
            else:
                # 3. Tenta conta 3.07 (Bancos)
                val_307 = grupo[grupo['CD_CONTA'] == '3.07'][col_valor].sum()
                if val_307 != 0:
                    ll = val_307
                else:
                    # 4. Fallback para 3.09
                    ll = grupo[grupo['CD_CONTA'] == '3.09'][col_valor].sum()
        
        rf = grupo[grupo['CD_CONTA'] == '3.06'][col_valor].sum()
        ir = grupo[grupo['CD_CONTA'] == '3.08'][col_valor].sum()
        
        resultados.append({
            'CNPJ_CLEAN': cnpj,
            'Lucro Liquido': ll,
            'Res_Fin': rf,
            'IR': ir
        })
    
    return pd.DataFrame(resultados)

def extrair_dre_trimestral(df_dre_trim, data):
    d = df_dre_trim[df_dre_trim['DT_FIM_EXERC'] == data]
    res = agrupar_contas_resultado(d, 'VL_REAL')
    if not d.empty and 'DT_INI_EXERC' in d.columns:
        dt_ini = d[['CNPJ_CLEAN', 'DT_INI_EXERC']].drop_duplicates().set_index('CNPJ_CLEAN')
        res = res.merge(dt_ini, left_on='CNPJ_CLEAN', right_index=True, how='left')
        res.rename(columns={'DT_INI_EXERC': 'DT_INI_REAL'}, inplace=True)
    return res

def agrupar_contas_fluxo(df_conta, col_valor):
    if 'DS_CONTA' in df_conta.columns:
        df_conta['DS_LOWER'] = df_conta['DS_CONTA'].astype(str).str.lower()
        mask = (df_conta['CD_CONTA'].str.startswith('6.01.01')) & \
               (df_conta['DS_LOWER'].str.contains('deprecia') | df_conta['DS_LOWER'].str.contains('amortiza'))
    else:
        mask = df_conta['CD_CONTA'].str.startswith('6.01.01')
    
    da = df_conta[mask].groupby('CNPJ_CLEAN')[col_valor].sum()
    return pd.DataFrame({'DA_Trimestral': da}).reset_index()

def extrair_dfc_trimestral(df_dfc, data):
    d = df_dfc[df_dfc['DT_FIM_EXERC'] == data].copy()
    if d.empty: return pd.DataFrame(columns=['CNPJ_CLEAN', 'DA_Trimestral'])
    return agrupar_contas_fluxo(d, 'VL_REAL')

def processar_ano_completo(ano):
    print(f"\n⚙️  Processando Ano {ano} (ITR + DFP)...")
    
    path_itr = download_zip_temporario('ITR', ano)
    path_dfp = download_zip_temporario('DFP', ano)

    cols_base = ['CNPJ_CIA', 'DENOM_CIA', 'DT_FIM_EXERC', 'CD_CONTA', 'DS_CONTA', 'VL_CONTA', 'ESCALA_MOEDA', 'ORDEM_EXERC']
    cols_fluxo = cols_base + ['DT_INI_EXERC']
    
    itr_dre = ler_csv_do_zip(path_itr, 'DRE_con', cols_fluxo)
    itr_dfc = ler_csv_do_zip(path_itr, 'DFC_MI_con', cols_fluxo)
    itr_bpa = ler_csv_do_zip(path_itr, 'BPA_con', cols_base)
    itr_bpp = ler_csv_do_zip(path_itr, 'BPP_con', cols_base)

    if itr_dre.empty: 
        print(f"   ⚠️ Sem dados ITR para {ano}")
        return pd.DataFrame()

    itr_dre['DIAS'] = (itr_dre['DT_FIM_EXERC'] - itr_dre['DT_INI_EXERC']).dt.days
    mask_trim = (itr_dre['DIAS'] >= 80) & (itr_dre['DIAS'] <= 100)
    dre_q123 = itr_dre[mask_trim].copy()
    
    mask_9m = (itr_dre['DT_INI_EXERC'].dt.month == 1) & (itr_dre['DT_FIM_EXERC'].dt.month == 9)
    dre_9m = itr_dre[mask_9m][['CNPJ_CLEAN', 'CD_CONTA', 'VL_REAL']].rename(columns={'VL_REAL': 'VL_9M'})
    
    dfc_9m = pd.DataFrame()
    if not itr_dfc.empty:
        mask_9m_dfc = (itr_dfc['DT_INI_EXERC'].dt.month == 1) & (itr_dfc['DT_FIM_EXERC'].dt.month == 9)
        dfc_9m = itr_dfc[mask_9m_dfc][['CNPJ_CLEAN', 'CD_CONTA', 'VL_REAL']].rename(columns={'VL_REAL': 'VL_9M'})

    dfp_dre = ler_csv_do_zip(path_dfp, 'DRE_con', cols_fluxo)
    dfp_dfc = ler_csv_do_zip(path_dfp, 'DFC_MI_con', cols_fluxo)
    dfp_bpa = ler_csv_do_zip(path_dfp, 'BPA_con', cols_base)
    dfp_bpp = ler_csv_do_zip(path_dfp, 'BPP_con', cols_base)
    
    resultados = []
    datas_itr = sorted(pd.to_datetime(itr_bpa['DT_FIM_EXERC'].unique()))
    
    for data in datas_itr:
        if data.month == 12: continue 
        
        patrimonio = extrair_patrimonio(itr_bpa, itr_bpp, data)
        if patrimonio.empty: continue
        
        dre_data = extrair_dre_trimestral(dre_q123, data)
        dfc_val = extrair_dfc_trimestral(itr_dfc, data)
        
        trimestre = patrimonio.merge(dre_data, on='CNPJ_CLEAN', how='left') \
                              .merge(dfc_val, on='CNPJ_CLEAN', how='left')
        
        trimestre['DT_FIM_EXERC'] = data
        trimestre['DT_INI_TRIM'] = data - timedelta(days=90)
        if not dre_data.empty and 'DT_INI_REAL' in dre_data.columns:
            trimestre['DT_INI_TRIM'] = dre_data['DT_INI_REAL']
            
        resultados.append(trimestre)

    if not dfp_dre.empty:
        dt_q4 = pd.Timestamp(f"{ano}-12-31")
        dt_ini_q4 = pd.Timestamp(f"{ano}-10-01")
        
        patrimonio_q4 = extrair_patrimonio(dfp_bpa, dfp_bpp, dt_q4)
        
        dfp_anual = dfp_dre[dfp_dre['DT_INI_EXERC'].dt.month == 1][['CNPJ_CLEAN', 'CD_CONTA', 'VL_REAL']]
        
        dre_calc = dfp_anual.merge(dre_9m, on=['CNPJ_CLEAN', 'CD_CONTA'], how='left')
        dre_calc['VL_9M'] = dre_calc['VL_9M'].fillna(0)
        dre_calc['VL_Q4'] = dre_calc['VL_REAL'] - dre_calc['VL_9M']
        
        dre_q4_final = agrupar_contas_resultado(dre_calc, 'VL_Q4')
        
        dfc_q4_final = pd.DataFrame(columns=['CNPJ_CLEAN', 'DA_Trimestral'])
        if not dfp_dfc.empty and not dfc_9m.empty:
            dfp_dfc_anual = dfp_dfc[dfp_dfc['DT_INI_EXERC'].dt.month == 1][['CNPJ_CLEAN', 'CD_CONTA', 'VL_REAL']]
            dfc_calc = dfp_dfc_anual.merge(dfc_9m, on=['CNPJ_CLEAN', 'CD_CONTA'], how='left')
            dfc_calc['VL_9M'] = dfc_calc['VL_9M'].fillna(0)
            dfc_calc['VL_Q4'] = dfc_calc['VL_REAL'] - dfc_calc['VL_9M']
            
            dfc_q4_final = agrupar_contas_fluxo(dfc_calc, 'VL_Q4')
            
        if not patrimonio_q4.empty:
            q4_df = patrimonio_q4.merge(dre_q4_final, on='CNPJ_CLEAN', how='left') \
                                 .merge(dfc_q4_final, on='CNPJ_CLEAN', how='left')
            
            q4_df['DT_FIM_EXERC'] = dt_q4
            q4_df['DT_INI_TRIM'] = dt_ini_q4
            resultados.append(q4_df)
    
    limpar_cache_ano(ano)
            
    if not resultados: return pd.DataFrame()
    df_final = pd.concat(resultados, ignore_index=True)
    
    nomes = itr_bpa[['CNPJ_CLEAN', 'DENOM_CIA', 'CNPJ_CIA']].drop_duplicates('CNPJ_CLEAN')
    df_final = df_final.merge(nomes, on='CNPJ_CLEAN', how='left')
    
    return df_final

# ==============================================================================
# 🚀 5. ENRIQUECIMENTO COM MERCADO (YAHOO FINANCE)
# ==============================================================================

def enriquecer_com_mercado(df, map_tickers, map_tipos):
    print(f"\n🌍 Consultando Preços de Fechamento Trimestral (Yahoo Finance)...")
    df['Ticker'] = df['CNPJ_CLEAN'].map(map_tickers)
    df['Tipo'] = df['CNPJ_CLEAN'].map(map_tipos)
    
    logging.getLogger('yfinance').setLevel(logging.CRITICAL)
    warnings.simplefilter('ignore', FutureWarning)
    
    df['EBITDA'] = df['Lucro Liquido'].fillna(0) - df['Res_Fin'].fillna(0) - df['IR'].fillna(0) + df['DA_Trimestral'].fillna(0)
    
    precos, mcaps, acoes_milhoes = [], [], []
    
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="   Download Preços", unit="emp"):
        t, dt_f = row['Ticker'], row['DT_FIM_EXERC']
        p_fechamento, mc, shares_mm = np.nan, np.nan, np.nan
        
        if pd.notna(t) and pd.notna(dt_f):
            t_sa = f"{t}.SA" if not t.endswith('.SA') else t
            try:
                start_win = dt_f - timedelta(days=15)
                end_win = dt_f + timedelta(days=1)
                
                with suppress_output():
                    h = yf.download(t_sa, start=start_win, end=end_win, progress=False, auto_adjust=True)
                
                if not h.empty:
                    val = h['Close'].iloc[-1]
                    if isinstance(val, pd.Series): val = val.iloc[0]
                    p_fechamento = float(val)
                    
                    tk = yf.Ticker(t_sa)
                    shares = tk.info.get('sharesOutstanding')
                    if shares:
                        mc = p_fechamento * shares
                        shares_mm = shares / 1_000_000 # Convertendo para Milhões
            except:
                pass
        
        precos.append(p_fechamento)
        mcaps.append(mc)
        acoes_milhoes.append(shares_mm)
        
    df['Preco_Fechamento'] = precos
    df['Market_Cap'] = mcaps
    df['Qtd_Acoes_Milhoes'] = acoes_milhoes
    
    # Cálculos
    df['EV'] = df['Market_Cap'].fillna(0) + df['Divida Liquida'].fillna(0)
    df['P_E'] = df['Market_Cap'] / (df['Lucro Liquido'] * 4) 
    df['EV_EBITDA'] = df['EV'] / (df['EBITDA'] * 4)
    df['Price_to_Book'] = df['Market_Cap'] / df['Patrimonio Liquido']
    
    # DL/EV (Evitar divisão por zero)
    df['DL_EV'] = df.apply(
        lambda row: row['Divida Liquida'] / row['EV'] if row['EV'] and row['EV'] != 0 else np.nan, 
        axis=1
    )
    
    warnings.simplefilter('default', FutureWarning)
    
    return df

# ==============================================================================
# 📌 6. RESUMOS (SETORES + MERCADO)
# ==============================================================================

def _safe_median(s):
    s = pd.to_numeric(s, errors="coerce")
    s = s.replace([np.inf, -np.inf], np.nan).dropna()
    if s.empty:
        return np.nan
    return s.median()

def _safe_sum(s):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan)
    return np.nan_to_num(s, nan=0.0).sum()

def _weighted_pe(market_cap, lucro):
    mc = pd.to_numeric(market_cap, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    ll = pd.to_numeric(lucro, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    ll_pos = ll.where(ll > 0, 0.0)
    denom = 4.0 * ll_pos.sum()
    if denom <= 0:
        return np.nan
    return mc.sum() / denom

def _weighted_ev_ebitda(ev, ebitda):
    evv = pd.to_numeric(ev, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    eb = pd.to_numeric(ebitda, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0.0)
    eb_pos = eb.where(eb > 0, 0.0)
    denom = 4.0 * eb_pos.sum()
    if denom <= 0:
        return np.nan
    return evv.sum() / denom

def gerar_resumos_snapshot(df):
    """
    Gera:
      - resumo_setores: 1 linha por Tipo (setor), usando o último DT_FIM_EXERC (snapshot)
      - resumo_mercado: 1 linha agregada, mesmo snapshot
    """
    if df.empty or "DT_FIM_EXERC" not in df.columns:
        return pd.DataFrame(), pd.DataFrame()

    snapshot_date = df["DT_FIM_EXERC"].max()
    snap = df[df["DT_FIM_EXERC"] == snapshot_date].copy()

    if "Tipo" not in snap.columns:
        snap["Tipo"] = np.nan

    snap["Tipo"] = snap["Tipo"].replace("", np.nan)

    # --- Resumo Setores (1 linha por Tipo)
    setores = (
        snap.dropna(subset=["Tipo"])
        .groupby("Tipo", dropna=True)
        .apply(lambda g: pd.Series({
            "DT_FIM_EXERC": snapshot_date,
            "N_Empresas": g["Ticker"].nunique() if "Ticker" in g.columns else len(g),
            "Market_Cap_Sum": _safe_sum(g["Market_Cap"]) if "Market_Cap" in g.columns else np.nan,
            "EV_Sum": _safe_sum(g["EV"]) if "EV" in g.columns else np.nan,
            "Lucro_Sum": _safe_sum(g["Lucro Liquido"]) if "Lucro Liquido" in g.columns else np.nan,
            "EBITDA_Sum": _safe_sum(g["EBITDA"]) if "EBITDA" in g.columns else np.nan,
            "Price_to_Book_Median": _safe_median(g["Price_to_Book"]) if "Price_to_Book" in g.columns else np.nan,
            "P_E_Median": _safe_median(g["P_E"]) if "P_E" in g.columns else np.nan,
            "EV_EBITDA_Median": _safe_median(g["EV_EBITDA"]) if "EV_EBITDA" in g.columns else np.nan,
            "P_E_Agregado": _weighted_pe(g.get("Market_Cap"), g.get("Lucro Liquido")),
            "EV_EBITDA_Agregado": _weighted_ev_ebitda(g.get("EV"), g.get("EBITDA")),
        }))
        .reset_index()
        .rename(columns={"Tipo": "SETOR"})
    )

    # --- Resumo Mercado (1 linha)
    mercado = pd.DataFrame([{
        "DT_FIM_EXERC": snapshot_date,
        "N_Empresas": snap["Ticker"].nunique() if "Ticker" in snap.columns else len(snap),
        "Market_Cap_Sum": _safe_sum(snap["Market_Cap"]) if "Market_Cap" in snap.columns else np.nan,
        "EV_Sum": _safe_sum(snap["EV"]) if "EV" in snap.columns else np.nan,
        "Lucro_Sum": _safe_sum(snap["Lucro Liquido"]) if "Lucro Liquido" in snap.columns else np.nan,
        "EBITDA_Sum": _safe_sum(snap["EBITDA"]) if "EBITDA" in snap.columns else np.nan,
        "Price_to_Book_Median": _safe_median(snap["Price_to_Book"]) if "Price_to_Book" in snap.columns else np.nan,
        "P_E_Median": _safe_median(snap["P_E"]) if "P_E" in snap.columns else np.nan,
        "EV_EBITDA_Median": _safe_median(snap["EV_EBITDA"]) if "EV_EBITDA" in snap.columns else np.nan,
        "P_E_Agregado": _weighted_pe(snap.get("Market_Cap"), snap.get("Lucro Liquido")),
        "EV_EBITDA_Agregado": _weighted_ev_ebitda(snap.get("EV"), snap.get("EBITDA")),
    }])

    return setores, mercado

# ==============================================================================
# 💾 7. EXPORT
# ==============================================================================

def gerar_excel(df):
    f = os.path.join(OUTPUT_DIR, f"Valuation_Final_{datetime.now().strftime('%Y%m%d')}.xlsx")
    print(f"\n💾 Salvando: {f}")
    
    wb = Workbook()
    
    header_fill = PatternFill(start_color="366092", end_color="366092", fill_type="solid")
    header_font = Font(color="FFFFFF", bold=True)
    
    # -----------------
    # Aba 1: Base Consolidada
    # -----------------
    ws = wb.active
    ws.title = "Base Consolidada"
    
    cols = ['CNPJ_CIA', 'DENOM_CIA', 'Ticker', 'Tipo', 'DT_FIM_EXERC',
            'Ativo Total', 'Caixa', 'Divida Bruta', 'Divida Liquida', 'Patrimonio Liquido',
            'Lucro Liquido', 'EBITDA', 'Preco_Fechamento', 'Qtd_Acoes_Milhoes', 
            'Market_Cap', 'EV', 'P_E', 'EV_EBITDA', 'Price_to_Book', 'DL_EV',
            'Res_Fin', 'IR', 'DA_Trimestral']
            
    df_s = df[[c for c in cols if c in df.columns]].copy()
    
    for r in dataframe_to_rows(df_s, index=False, header=True):
        ws.append(r)
        
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
    
    # Formatação DL/EV como percentual
    col_dlev = None
    for idx, cell in enumerate(ws[1], 1):
        if cell.value == 'DL_EV':
            col_dlev = idx
            break
    if col_dlev:
        for row in ws.iter_rows(min_row=2, min_col=col_dlev, max_col=col_dlev):
            for cell in row:
                cell.number_format = '0.00%'

    # -----------------
    # Aba 2: Resumo Setores (snapshot)
    # -----------------
    resumo_setores, resumo_mercado = gerar_resumos_snapshot(df)

    ws2 = wb.create_sheet("Resumo_Setores")
    if not resumo_setores.empty:
        for r in dataframe_to_rows(resumo_setores, index=False, header=True):
            ws2.append(r)
        for cell in ws2[1]:
            cell.fill = header_fill
            cell.font = header_font
    else:
        ws2.append(["Sem dados de setor (coluna 'TIPO' ausente ou sem match)."])

    # -----------------
    # Aba 3: Resumo Mercado (snapshot)
    # -----------------
    ws3 = wb.create_sheet("Resumo_Mercado")
    if not resumo_mercado.empty:
        for r in dataframe_to_rows(resumo_mercado, index=False, header=True):
            ws3.append(r)
        for cell in ws3[1]:
            cell.fill = header_fill
            cell.font = header_font
    else:
        ws3.append(["Sem dados para resumo de mercado."])

    wb.save(f)
    print("✅ Concluído.")

# ==============================================================================
# MAIN
# ==============================================================================

if __name__ == "__main__":
    setup_directories()
    
    anos_processar = range(ANO_INICIO, ANO_FIM + 1)
    
    dfs = []
    for a in anos_processar:
        df_ano = processar_ano_completo(a)
        if not df_ano.empty: dfs.append(df_ano)
        
    if dfs:
        full_df = pd.concat(dfs, ignore_index=True)
        # Agora retorna 2 dicionários
        map_tickers, map_tipos = carregar_tickers_e_tipos(CAMINHO_TICKERS)
        
        if map_tickers:
            final_df = enriquecer_com_mercado(full_df, map_tickers, map_tipos)
            gerar_excel(final_df)
        else:
            print("❌ Sem tickers, abortando valuation.")
    else:
        print("❌ Nenhum dado processado.")